# Module 2.2 — Tool Use Evaluation

Tool use is the heart of any LLM agent. Regardless of how well an agent reasons or plans, if it calls the wrong function, passes malformed arguments, or invokes tools in the wrong sequence, the task fails. And here is the tricky part: the final text output can look perfectly plausible even when the tool execution underneath was completely broken. A travel bot that confidently says "Your flight is booked!" is useless if it never actually called the booking API.

This makes tool use evaluation fundamentally different from standard LLM evaluation, and arguably the most important dimension for agentic systems.

_Source: `multi-turn eval and tool evaluations/evaluation.ipynb`, Part 2 (unmodified)._

### Why tool use evaluation requires its own approach

In a standard LLM evaluation, you feed a prompt to a model and compare the output against a reference answer. With tool-using systems, the output is only the tip of the iceberg. Beneath it lies a sequence of decisions: which tools to invoke, in what order, with what arguments, and how to interpret the results. Each of these decisions is a potential failure point, and each requires its own evaluation.

Consider a customer support agent with access to three tools: `OrderLookup`, `PolicyRetriever`, and `RefundProcessor`. A user asks about returning an item from order #12345. The correct behavior is to first look up the order, then retrieve the return policy, and finally process the refund if eligible.

But it might happen that the agent skips the lookup and hallucinates order details. Or it might call the right tools but pass the wrong order ID. Or it might call `RefundProcessor` before checking whether the item is even eligible. The final response could sound correct in all these cases, but only a tool-level evaluation would catch the underlying failures.

Tool evaluation therefore asks a specific set of questions that traditional evaluation ignores:
- Did the agent select the right tools?
- Did the agent call tools in the correct order?
- Did the agent supply correct arguments?

DeepEval provides three dedicated metrics for evaluating tool use, each targeting a different dimension and operating at a different level of granularity:
- **`ToolCorrectnessMetric`** — reference-based, compares `tools_called` against `expected_tools`. Great for regression/CI when you already know the correct tool behavior.
- **`ArgumentCorrectnessMetric`** — referenceless, LLM-judged evaluation of whether the arguments passed to each tool call were correct given the task. Ideal for dynamic workflows and production monitoring where exact argument values can't be predetermined.
- **`ToolUseMetric`** — multi-turn version, operates on a `ConversationalTestCase`. Produces a tool selection score and an argument correctness score, with the final score being the minimum of both.

### `ToolCorrectnessMetric`: reference-based tool evaluation metric

The `ToolCorrectnessMetric` is the most direct tool evaluation metric. It compares the tools actually called (`tools_called`) against a known expected set (`expected_tools`). This also makes it a reference-based metric, i.e., you must know in advance which tools should have been called.

This example shows how to evaluate whether the right tools were used in the right way during a task. Instead of judging the final text output, it checks the actual tool usage against what we expected it to do:
- `ToolCall` represents an individual tool details/invocation, and `ToolCallParams` lets you specify what parts of each tool call should be compared during evaluation. `ToolCorrectnessMetric` is the metric that performs the comparison.
- The `available_tools` list defines the tools that were theoretically available to the agent. Each tool is described with a name, input schema, and a short description. This list is optional, but if provided, DeepEval can additionally judge whether the agent actually chose the most appropriate tools from all possible options.
- The `expected_tools` section within `test_case` defines the reference behavior.
- Finally, the `ToolCorrectnessMetric` configuration controls how strict the evaluation should be. By including `ToolCallParams.INPUT_PARAMETERS`, the metric checks whether the tool arguments match. By including `ToolCallParams.OUTPUT`, it also checks whether the recorded outputs align.
- The `should_exact_match=True` setting makes the evaluation stricter. It requires the actual tool calls and expected tool calls to match exactly. That means extra tools, missing tools, mismatched details or wrong order of tool calling can be penalized.
- The commented-out `available_tools=available_tools` line shows an optional enhancement. If enabled, DeepEval can use an LLM judge to determine whether the chosen tools were the optimal choices from the set of available tools. In that mode, the final score becomes more conservative, because it combines the deterministic matching with an LLM-based judgment of tool selection quality.

Overall, this metric is great for development, testing, and regression checks. It works especially well when you already know what the correct tool behavior should look like.

In [ ]:
# ============ TOOLCORRECTNESSMETRIC ============
from deepeval.test_case import LLMTestCase, ToolCall, ToolCallParams
from deepeval.metrics import ToolCorrectnessMetric
from dotenv import load_dotenv
load_dotenv()

available_tools = [
    ToolCall(name="FlightSearch",
             input_parameters={"origin": "str",
                               "destination": "str", "date": "str"},
             description="Search flights tool"),
    ToolCall(name="WeatherCheck",
             input_parameters={"city": "str", "date": "str"},
             description="Check weather for a city"),
    ToolCall(name="HotelSearch",
             input_parameters={"city": "str",
                               "check_in": "str", "check_out": "str"},
             description="Search hotels in a city"),
    ToolCall(name="CurrencyConvert",
             input_parameters={"from": "str", "to": "str", "amount": "float"},
             description="Convert from one currency to another"),
    ToolCall(name="WeatherCheckV2",
             input_parameters={"city": "str", "date": "str"},
             description="Newer weather tool"),
]

test_case = LLMTestCase(
    input="Find me flights from NYC to London on 2026-03-13 and check the weather there.",
    actual_output="Found 2 flights from NYC to London. Weather in London: 12°C, cloudy.",

    tools_called=[
        ToolCall(
            name="FlightSearch",
            input_parameters={"origin": "NYC", "destination": "London",
                              "date": "2026-03-13"},
            output={"flights":
                    [{"id": "BA178", "price_usd": 412}, {
                        "id": "VS4", "price_usd": 389}],
                    "total": 2}
        ),
        ToolCall(
            name="WeatherCheck",
            input_parameters={"city": "London", "date": "2026-03-13"},
        ),
    ],

    expected_tools=[
        ToolCall(
            name="FlightSearch",
            input_parameters={"origin": "NYC", "destination": "London",
                              "date": "2026-03-13"},
            output={"flights":
                    [{"id": "BA178", "price_usd": 412}, {
                        "id": "VS4", "price_usd": 389}],
                    "total": 2}
        ),
        ToolCall(
            name="WeatherCheck",
            input_parameters={"city": "London", "date": "2026-03-13"},
        ),
    ],
)

metric = ToolCorrectnessMetric(
    evaluation_params=[
        ToolCallParams.INPUT_PARAMETERS,
        ToolCallParams.OUTPUT,
    ],
    should_exact_match=True,
    available_tools=available_tools,
    # model="gpt-4o",
    include_reason=True,
)

metric.measure(test_case)
print(f"Score: {metric.score}")
print(f"Reason: {metric.reason}")

### `ArgumentCorrectnessMetric`: referenceless argument evaluation

The `ArgumentCorrectnessMetric` focuses specifically on whether the arguments (input parameters) passed to each tool call were correct for the given task. Unlike `ToolCorrectnessMetric`, this metric is fully referenceless and LLM-judged: it evaluates argument quality based on the user's input and the tool descriptions, without requiring expected argument values.

This example evaluates whether the right arguments are passed to a tool (but with an extra safety layer). It combines a manual schema validation step with DeepEval's `ArgumentCorrectnessMetric`. The manual validation catches obvious structural mistakes first, and only if that passes does the LLM-based metric run:
- The `TOOL_SCHEMAS` dictionary defines the expected argument structure for the `FlightSearch` tool. It says that this tool must receive exactly three keys: `origin`, `destination`, and `date`. It also defines a validator for the date field, where `datetime.strptime(v, "%Y-%m-%d")` is used to check that the value is a properly formatted date string.
- The `validate_args()` function is a deterministic guardrail. It loops through all tool calls made by the agent and checks whether a schema exists for that tool. If no schema is defined, it skips validation for that tool. For tools that do have a schema, it compares the provided argument keys against the required ones. And after the key checks, the function validates field formats. If all tools pass these checks, the function returns `True, "ok"`. That means the tool arguments are structurally valid and the script can proceed to the LLM-based evaluation.
- The `ArgumentCorrectnessMetric` itself is then defined with a threshold, an evaluation model, and `include_reason=True`. This metric is meant to judge whether the arguments passed to the tool are appropriate given the user's request, the tool description, and the overall context.
- The control flow at the end is important. First, `validate_args(test_case.tools_called)` is run. If that check fails, the script immediately prints `Score: 0` and the deterministic reason — the LLM metric is skipped entirely, because there is no point in evaluating clearly malformed tool call(s). If the validation passes, the script then calls `metric.measure(test_case)`. At that point, DeepEval uses the LLM judge to assess the correctness of the arguments more intelligently. The final printed score and reason then come from the metric rather than the manual validator.

So the overall pattern here is: first enforce hard input constraints deterministically, then check semantic correctness of arguments with DeepEval. This is a strong setup for agent evaluation because it separates the two kinds of correctness.

`ArgumentCorrectnessMetric`, owing to its nature, is ideal for two scenarios: dynamic workflows where the exact argument values cannot be predetermined, and production monitoring, since it does not require labeled data to produce a meaningful score.

In [ ]:
# ============ ARGUMENTCORRECTNESSMETRIC ============
from deepeval.metrics import ArgumentCorrectnessMetric
from deepeval.test_case import LLMTestCase, ToolCall
from datetime import datetime
from dotenv import load_dotenv
load_dotenv()

TOOL_SCHEMAS = {
    "FlightSearch": {
        "required_keys": {"origin", "destination", "date"},
        "validators": {
            "date": (lambda v: datetime.strptime(v, "%Y-%m-%d"), "YYYY-MM-DD")
        }
    }
}


def validate_args(tools_called: list[ToolCall]) -> tuple[bool, str]:
    for tool in tools_called:
        schema = TOOL_SCHEMAS.get(tool.name)
        if not schema:
            continue

        actual_keys = set(tool.input_parameters.keys())
        required_keys = schema["required_keys"]

        missing = required_keys - actual_keys
        if missing:
            return False, f"{tool.name}: missing required args {missing}"

        extra = actual_keys - required_keys
        if extra:
            return False, f"{tool.name}: unexpected args {extra}"

        for field, (validator, fmt) in schema["validators"].items():
            value = tool.input_parameters.get(field)
            try:
                validator(value)
            except (ValueError, TypeError):
                return False, f"{tool.name}: '{field}' must be '{fmt}', got '{value}'"

    return True, "ok"


metric = ArgumentCorrectnessMetric(
    threshold=0.7,
    model="gpt-4o",
    include_reason=True,
)

test_case = LLMTestCase(
    input="Find me flights from New York to Paris on March 15, 2026",
    actual_output="I found 3 flights from NYC to Paris on March 15.",
    tools_called=[
        ToolCall(
            name="FlightSearch",
            description="Search for available flights between cities on a given date (date in YYYY-MM-DD format).",
            input_parameters={"origin": "NYC",
                              "destination": "Paris", "date": "2026-03-15"}
        ),
    ],
)

valid, reason = validate_args(test_case.tools_called)
if not valid:
    print(f"Score: 0\nReason: {reason}")
else:
    metric.measure(test_case)
    print(f"Score: {metric.score}\nReason: {metric.reason}")

### `ToolUseMetric`: multi-turn tool evaluation

The `ToolUseMetric` evaluates tool usage across a multi-turn conversation. It operates on a `ConversationalTestCase`, making it the right choice for chatbot-style agents where tool use can span multiple exchanges.

The `ToolUseMetric` produces two sub-scores: a tool selection score and an argument correctness score. The final score is the minimum of both, so a failure in either dimension pulls the overall score down.

The `available_tools` parameter is required. By telling the metric which tools the agent could have used, you enable it to evaluate whether each selection was optimal given the alternatives.

This metric is most useful for evaluating conversational agents where tool use decisions depend on evolving context.

To summarize, the three metrics serve three key purposes:
- Use `ToolCorrectnessMetric` when you have labeled test cases with known expected tools. This is a great choice for regression testing and CI/CD pipelines.
- Use `ArgumentCorrectnessMetric` when you need to evaluate argument quality without reference data. This is the right choice for dynamic workflows, and cases where the exact argument values depend on runtime context.
- Use `ToolUseMetric` when evaluating multi-turn conversations where tool use spans multiple exchanges.

In [ ]:
# ============ TOOLUSEMETRIC ============
from deepeval.metrics import ToolUseMetric
from deepeval.test_case import Turn, ConversationalTestCase, ToolCall
from dotenv import load_dotenv
load_dotenv()

convo_test_case = ConversationalTestCase(
    turns=[
        Turn(role="user", content="Find me a hotel in Paris for March 15-18, 2026"),
        Turn(
            role="assistant",
            content="I found one option. The Hotel Le Marais (id: 'le-marais-paris') has availability at $180/night.",
            tools_called=[
                ToolCall(
                    name="HotelSearch",
                    input_parameters={
                        "city": "Paris", "checkin": "2026-03-15", "checkout": "2026-03-18"},
                    output={"hotel_id": "le-marais-paris",
                            "name": "Le Marais", "price_usd": 180}
                )
            ]
        ),
        Turn(role="user", content="Yes, book the same please (id: 'le-marais-paris'), from 15 to 18 March."),
        Turn(
            role="assistant",
            content="Done! Your reservation is confirmed. Confirmation: HTL-9921.",
            tools_called=[
                ToolCall(
                    name="HotelBooking",
                    input_parameters={"hotel_id": "le-marais-paris",
                                      "checkin": "2026-03-15", "checkout": "2026-03-18"},
                    output={"confirmation": "HTL-9921"}
                )
            ]
        ),
    ],
)

metric = ToolUseMetric(
    model="gpt-4o",
    include_reason=True,
    available_tools=[
        ToolCall(name="HotelSearch",
                 description="Search for hotels by city and dates"),
        ToolCall(name="HotelBooking",
                 description="Book a hotel by hotel ID and dates"),
        ToolCall(name="FlightSearch",
                 description="Search for flights between cities"),
        ToolCall(name="CarRental", description="Search for rental cars"),
    ],
    # strict_mode=True,
)

metric.measure(convo_test_case)
print(f"Score: {metric.score}")
print(f"Reason: {metric.reason}")

## Summary

- **`ToolCorrectnessMetric`** — reference-based, compares `tools_called` against `expected_tools`. Best for regression/CI when you already know the correct tool behavior.
- **`ArgumentCorrectnessMetric`** — referenceless, LLM-judged, ideal for dynamic workflows and production monitoring.
- **`ToolUseMetric`** — multi-turn, combines tool selection and argument correctness via `min()`.
- All three judge the *mechanics* of tool use (right tools, right arguments, right order) — none of them ask whether the agent actually accomplished the task. That gap is what Module 2.3 covers next.
- Next: [Module 2.3](10_Task_Completion_Evaluation.ipynb) — task completion and outcome verification.